In [ ]:

!pip install opencv-python


EXTRACTING DATA
::::::::::::::::::::::::::::::
::::::::::::::::::::::::::::::

In [ ]:
#extract from dataset
import os

your_dataset = []
base_folder = "GTSRB/Train"  # path to your Train folder

for label_folder in os.listdir(base_folder):
    label_path = os.path.join(base_folder, label_folder)
    if os.path.isdir(label_path):
        for filename in os.listdir(label_path):
            if filename.endswith(".ppm") or filename.endswith(".png") or filename.endswith(".jpg"):
                img_path = os.path.join(label_path, filename)
                your_dataset.append((img_path, int(label_folder)))  # label is the folder name as integer




NORMAL LOGISTIC REGRESSION
::::::::::::::::::::::::::
:::::::::::::::::::::::::::


In [ ]:
#logistic regression model train
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score, accuracy_score, precision_score
import numpy as np
import cv2
import os
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
X = [] 
y = [] 

for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64,64))
    X.append(img.flatten() / 255.0)
    y.append(label)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(multi_class='multinomial', max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred = model.predict(X_test)
#report = classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)])
#print(report)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"Weighted Precision: {precision:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


FIRST TEST
:::::::::::::::::::;
::::::::::::::::::::

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
test_folder = "../RoadsignRecognition/GTSRB/Test"
for filename in os.listdir(test_folder):
    if filename.endswith(".ppm") or filename.endswith(".png") or filename.endswith(".jpg"):
        img_path = os.path.join(test_folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64,64))
        img_flattened = img.flatten() / 255.0
        predicted_label = model.predict([img_flattened])
        print(f"Image: {filename}, Predicted Label: {predicted_label[0]},label_map[predicted_label]")
print()

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()
plt.figure(figsize=(14, 5))
sns.barplot(x=class_names, y=class_accuracies)
plt.title("Per-Class Accuracy")
plt.ylabel("Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# --- Print: Top Confused Pairs ---
#


ACTUAL TEST
:::::::::::::::::
:::::::::::::::::


In [ ]:
# needed
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2

# Label map (minimal or full version; update as needed)
label_map = {
    0: "Speed limit 20", 1: "Speed limit 30", 2: "Speed limit 50", 3: "Speed limit 60", 4: "Speed limit 70",
    5: "Speed limit 80", 6: "End limit 80", 7: "Speed limit 100", 8: "Speed limit 120", 9: "No overtaking",
    10: "No overtaking (trucks)", 11: "Right of way", 12: "Yield", 13: "Stop", 14: "No vehicles",
    15: "No trucks", 16: "No entry", 17: "Danger", 18: "Left curve", 19: "Right curve",
    20: "Double curve", 21: "Uneven road", 22: "Slippery", 23: "Road narrows", 24: "Construction",
    25: "Signal", 26: "Pedestrian", 27: "Children", 28: "Bicycles", 29: "Ice/Snow",
    30: "Animals", 31: "End restrictions", 32: "Turn right", 33: "Turn left", 34: "Go straight",
    35: "Go straight/right", 36: "Go straight/left", 37: "Keep right", 38: "Keep left", 39: "Roundabout",
    40: "End no overtaking", 41: "End no overtaking (trucks)", 42: "Other"
}
class_names = [label_map[i] for i in range(43)]

# --- Predict on External Images ---
test_folder = "../RoadsignRecognition/GTSRB/Test"
print("Individual Image Predictions:\n")
for filename in os.listdir(test_folder):
    if filename.lower().endswith((".ppm", ".png", ".jpg")):
        img_path = os.path.join(test_folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64,64))
        img_flattened = img.flatten() / 255.0
        predicted_label = model.predict([img_flattened])[0]
        print(f"Image: {filename}, Predicted Label: {predicted_label}, Class: {label_map[predicted_label]}")
print()

# --- Confusion Matrix ---
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# --- Per-Class Accuracy ---
correct_per_class = cm.diagonal()
total_per_class = cm.sum(axis=1)
class_accuracies = correct_per_class / total_per_class

plt.figure(figsize=(14, 5))
sns.barplot(x=class_names, y=class_accuracies)
plt.title("Per-Class Accuracy")
plt.ylabel("Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


DATA AUGMENTATION AND LOGISTIC REGRESSION
::::::::::::::::::::::::::
::::::::::::::::::::::::::

In [ ]:
import numpy as np
import cv2

# Weather effect functions
def add_fog(image, intensity=0.9):
    h, w = image.shape
    fog = np.random.normal(loc=240, scale=25, size=(h, w)).astype(np.uint8)
    fog = cv2.GaussianBlur(fog, (151, 151), 0)  # large blur kernel
    foggy = cv2.addWeighted(image, 1 - intensity, fog, intensity, 0)
    return np.clip(foggy, 0, 255).astype(np.uint8)




def add_rain(image, drops=100):
    rain_img = image.copy()
    for _ in range(drops):
        x = np.random.randint(0, image.shape[1])
        y = np.random.randint(0, image.shape[0])
        length = np.random.randint(5, 15)
        cv2.line(rain_img, (x, y), (x, y + length), (255,), 1)
    return cv2.blur(rain_img, (3, 3))
def night_effect(image, factor=0.4):
    h, w = image.shape
    X_resultant_kernel = cv2.getGaussianKernel(w, 100)
    Y_resultant_kernel = cv2.getGaussianKernel(h, 100)
    kernel = Y_resultant_kernel * X_resultant_kernel.T
    mask = 255 * kernel / np.linalg.norm(kernel)
    night = image * factor * mask
    return np.clip(night, 0, 255).astype(np.uint8)



def add_glare(image, strength=0.6):
    overlay = image.copy()
    h, w = image.shape
    center = (np.random.randint(0, w), np.random.randint(0, h))
    radius = np.random.randint(30, 60)
    cv2.circle(overlay, center, radius, (255,), -1)
    return cv2.addWeighted(image, 1 - strength, overlay, strength, 0)

# Apply a weather effect batch-wise
def apply_effect_batch(X, effect_fn):
    transformed = []
    for x in X:
        img = (x * 255).reshape(64, 64).astype(np.uint8)
        effected = effect_fn(img)
        effected_flat = effected.flatten() / 255.0
        transformed.append(effected_flat)
    return np.array(transformed)

# Assuming you already have X_test and y_test from your dataset split
X_test_foggy = apply_effect_batch(X_test, add_fog)
X_test_rainy = apply_effect_batch(X_test, add_rain)
X_test_night = apply_effect_batch(X_test, night_effect)
X_test_glare = apply_effect_batch(X_test, add_glare)

# Predict and evaluate
from sklearn.metrics import accuracy_score

y_pred_normal = model.predict(X_test)
y_pred_foggy = model.predict(X_test_foggy)
y_pred_rainy = model.predict(X_test_rainy)
y_pred_night = model.predict(X_test_night)
y_pred_glare = model.predict(X_test_glare)

acc_normal = accuracy_score(y_test, y_pred_normal)
acc_foggy = accuracy_score(y_test, y_pred_foggy)
acc_rainy = accuracy_score(y_test, y_pred_rainy)
acc_night = accuracy_score(y_test, y_pred_night)
acc_glare = accuracy_score(y_test, y_pred_glare)

print("Normal Accuracy:", acc_normal)
print("Foggy Accuracy:", acc_foggy)
print("Rainy Accuracy:", acc_rainy)
print("Night Accuracy:", acc_night)
print("Glare Accuracy:", acc_glare)


AUGMENTATION DATA
::::::::::::::::::::
::::::::::::::::::::::

In [ ]:
import matplotlib.pyplot as plt

# Choose a random index from test set
idx = np.random.randint(0, len(X_test))
original_img = (X_test[idx] * 255).reshape(64, 64).astype(np.uint8)

# Apply weather effects
foggy_img = add_fog(original_img)
rainy_img = add_rain(original_img)
night_img = night_effect(original_img)
glare_img = add_glare(original_img)

# Plot all images
titles = ["Original", "Foggy", "Rainy", "Night", "Glare"]
images = [original_img, foggy_img, rainy_img, night_img, glare_img]

plt.figure(figsize=(15, 3))
for i, (title, img) in enumerate(zip(titles, images)):
    plt.subplot(1, 5, i+1)
    plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
plt.tight_layout()
plt.show()


FAILED LOGISTIC REGRESSION
::::::::::::::::::::::::::::
:::::::::::::::::::::::::::::

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# --- Weather Effect Functions (Same as before) ---
def add_fog(image, intensity=0.9):
    h, w = image.shape
    fog = np.random.normal(loc=240, scale=25, size=(h, w)).astype(np.uint8)
    fog = cv2.GaussianBlur(fog, (151, 151), 0)
    foggy = cv2.addWeighted(image, 1 - intensity, fog, intensity, 0)
    return np.clip(foggy, 0, 255).astype(np.uint8)

def add_rain(image, drops=100):
    rain_img = image.copy()
    for _ in range(drops):
        x = np.random.randint(0, image.shape[1])
        y = np.random.randint(0, image.shape[0])
        length = np.random.randint(5, 15)
        cv2.line(rain_img, (x, y), (x, y + length), (220,), 1)
    return cv2.blur(rain_img, (3, 3))

def night_effect(image, factor=0.4):
    h, w = image.shape
    X_resultant_kernel = cv2.getGaussianKernel(w, 100)
    Y_resultant_kernel = cv2.getGaussianKernel(h, 100)
    kernel = Y_resultant_kernel * X_resultant_kernel.T
    mask = 255 * kernel / np.linalg.norm(kernel)
    night = image.astype(float) * factor * (mask / 255.0)
    return np.clip(night, 0, 255).astype(np.uint8)

def add_glare(image, strength=0.6):
    overlay = image.copy()
    h, w = image.shape
    center = (np.random.randint(0, w), np.random.randint(0, h))
    radius = np.random.randint(30, 60)
    cv2.circle(overlay, center, radius, (255,), -1)
    overlay = cv2.GaussianBlur(overlay, (99,99), 0)
    return cv2.addWeighted(image, 1 - strength, overlay, strength, 0)

# --- Batch Application Function (Same as before) ---
def apply_effect_batch(X, effect_fn):
    transformed = []
    for x in X:
        img = (x * 255).reshape(64, 64).astype(np.uint8)
        effected = effect_fn(img)
        effected_flat = effected.flatten() / 255.0
        transformed.append(effected_flat)
    return np.array(transformed)


# --- Step 1: Load REAL GTSRB Data (Your Code) ---
print("Step 1: Loading GTSRB data from disk...")
your_dataset = []
base_folder = "GTSRB/Train"  # <--- MAKE SURE THIS PATH IS CORRECT

for label_folder in os.listdir(base_folder):
    label_path = os.path.join(base_folder, label_folder)
    if os.path.isdir(label_path):
        for filename in os.listdir(label_path):
            if filename.endswith(".ppm"): # GTSRB uses .ppm files
                img_path = os.path.join(label_path, filename)
                your_dataset.append((img_path, int(label_folder)))

X_data = []
y_data = []

for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = cv2.resize(img, (64, 64))
        X_data.append(img.flatten() / 255.0)
        y_data.append(label)

X = np.array(X_data)
y = np.array(y_data)
print(f"Loaded {len(X)} images. Data shape: {X.shape}")
print("-" * 30)


# --- Step 2: Split Data into Training and Testing Sets ---
print("Step 2: Splitting data into training and test sets...")
# Using stratify=y is important for imbalanced datasets like GTSRB
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Data shapes: X_train: {X_train.shape}, X_test: {X_test.shape}")
print("-" * 30)


# --- Step 3: Create Augmented Datasets for Training and Testing ---
print("Step 3: Augmenting data with weather effects...")
# Augment the TRAINING data
X_train_foggy = apply_effect_batch(X_train, add_fog)
X_train_rainy = apply_effect_batch(X_train, add_rain)

# Augment the TEST data
X_test_foggy = apply_effect_batch(X_test, add_fog)
X_test_rainy = apply_effect_batch(X_test, add_rain)
X_test_night = apply_effect_batch(X_test, night_effect)
X_test_glare = apply_effect_batch(X_test, add_glare)
print("Data augmentation complete.")
print("-" * 30)


# --- Step 4: Combine Datasets for Robust Training ---
print("Step 4: Combining original and augmented training data...")
X_train_combined = np.vstack([X_train, X_train_foggy, X_train_rainy])
y_train_combined = np.concatenate([y_train, y_train, y_train])

# Shuffle the combined dataset
shuffle_idx = np.random.permutation(len(X_train_combined))
X_train_combined = X_train_combined[shuffle_idx]
y_train_combined = y_train_combined[shuffle_idx]

print(f"Combined training data shape: {X_train_combined.shape}")
print("-" * 30)


# --- Step 5: Train the ROBUST Logistic Regression Model ---
print("Step 5: Training the robust Logistic Regression model...")
# INCREASED max_iter to prevent ConvergenceWarning on the larger dataset
lr_model = LogisticRegression(
    multi_class='multinomial',
    max_iter=3000, # Increased from 1000
    random_state=42,
    solver='saga' # Good solver for large datasets
)
lr_model.fit(X_train_combined, y_train_combined)
print("Model training complete.")
print("-" * 30)


# --- Step 6: Evaluate the New Model on All Test Conditions ---
print("Step 6: Evaluating the new model...")
y_pred_normal = lr_model.predict(X_test)
y_pred_foggy = lr_model.predict(X_test_foggy)
y_pred_rainy = lr_model.predict(X_test_rainy)
y_pred_night = lr_model.predict(X_test_night)
y_pred_glare = lr_model.predict(X_test_glare)

accuracies = {
    "Normal": accuracy_score(y_test, y_pred_normal),
    "Foggy": accuracy_score(y_test, y_pred_foggy),
    "Rainy": accuracy_score(y_test, y_pred_rainy),
    "Night": accuracy_score(y_test, y_pred_night),
    "Glare": accuracy_score(y_test, y_pred_glare)
}

print("--- Robust Model Performance ---")
for condition, acc in accuracies.items():
    print(f"{condition} Accuracy: {acc:.4f}")
print("-" * 30)


# --- Step 7: Visualization ---
print("Step 7: Generating visualizations...")

# Accuracy Bar Chart
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 6))
conditions = list(accuracies.keys())
scores = list(accuracies.values())
bars = ax.bar(conditions, scores, color=['#4CAF50', '#FFC107', '#2196F3', '#3F51B5', '#FF5722'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy Score', fontsize=12)
ax.set_title('Robust Logistic Regression Performance on GTSRB', fontsize=16, pad=20)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.01, f'{yval:.3f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

# Confusion Matrix Heatmap (for the 'Normal' test case)
cm = confusion_matrix(y_test, y_pred_normal)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=range(43), yticklabels=range(43))
plt.title('Confusion Matrix on Normal (Unaffected) Test Data', fontsize=16, pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.show()

FAILED AUGMENTED LOGISTIC REGRESSION
::::::::::::::::::::::::::::::::::::
::::::::::::::::::::::::::::::::::::

In [ ]:
# --- Block 1: Imports ---
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# --- Block 2: Weather Effect Functions ---
def add_fog(image, intensity=0.9):
    """Adds a dense fog effect to a grayscale image."""
    h, w = image.shape
    fog = np.random.normal(loc=240, scale=25, size=(h, w)).astype(np.uint8)
    fog = cv2.GaussianBlur(fog, (151, 151), 0)
    foggy = cv2.addWeighted(image, 1 - intensity, fog, intensity, 0)
    return np.clip(foggy, 0, 255).astype(np.uint8)

def add_rain(image, drops=100):
    """Adds rain streaks to a grayscale image."""
    rain_img = image.copy()
    for _ in range(drops):
        x = np.random.randint(0, image.shape[1])
        y = np.random.randint(0, image.shape[0])
        length = np.random.randint(5, 15)
        cv2.line(rain_img, (x, y), (x, y + length), (220,), 1)
    return cv2.blur(rain_img, (3, 3))

def night_effect(image, factor=0.4):
    """Simulates a night-time view with a vignette effect."""
    h, w = image.shape
    X_resultant_kernel = cv2.getGaussianKernel(w, 100)
    Y_resultant_kernel = cv2.getGaussianKernel(h, 100)
    kernel = Y_resultant_kernel * X_resultant_kernel.T
    mask = 255 * kernel / np.linalg.norm(kernel)
    night = image.astype(float) * factor * (mask / 255.0)
    return np.clip(night, 0, 255).astype(np.uint8)

def add_glare(image, strength=0.6):
    """Adds a bright, blurry glare spot to a grayscale image."""
    overlay = image.copy()
    h, w = image.shape
    center = (np.random.randint(0, w), np.random.randint(0, h))
    radius = np.random.randint(30, 60)
    cv2.circle(overlay, center, radius, (255,), -1)
    overlay = cv2.GaussianBlur(overlay, (99,99), 0)
    return cv2.addWeighted(image, 1 - strength, overlay, strength, 0)

# --- Block 3: Batch Application Helper Function ---
def apply_effect_batch(X, effect_fn):
    """Applies a given effect function to a batch of flattened images."""
    transformed = []
    print(f"Applying effect: {effect_fn.__name__}...")
    for x in X:
        img = (x * 255).reshape(64, 64).astype(np.uint8)
        effected = effect_fn(img)
        effected_flat = effected.flatten() / 255.0
        transformed.append(effected_flat)
    return np.array(transformed)

# --- Block 4: Load REAL GTSRB Data with Error Checking ---
print("--- Step 1: Loading GTSRB data from disk ---")
your_dataset = []
base_folder = "GTSRB/Train"  # Ensure this path is correct

if not os.path.isdir(base_folder):
    print(f"ERROR: The folder '{base_folder}' was not found.")
    print("Please make sure the path is correct or the script is in the right directory.")
    sys.exit()

found_files = False
for label_folder in os.listdir(base_folder):
    label_path = os.path.join(base_folder, label_folder)
    if os.path.isdir(label_path):
        for filename in os.listdir(label_path):
            if filename.lower().endswith(('.ppm', '.png', '.jpg', '.jpeg')):
                found_files = True
                img_path = os.path.join(label_path, filename)
                try:
                    your_dataset.append((img_path, int(label_folder)))
                except ValueError:
                    # Ignore folders with non-integer names (like .DS_Store)
                    continue

if not found_files:
    print(f"ERROR: No image files (.ppm, .png, etc.) found in the subdirectories of '{base_folder}'.")
    sys.exit()

X_data = []
y_data = []
for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = cv2.resize(img, (64, 64))
        X_data.append(img.flatten() / 255.0)
        y_data.append(label)

X = np.array(X_data)
y = np.array(y_data)
print(f"Successfully loaded {len(X)} images. Data shape: {X.shape}")
print("-" * 30)

# --- Block 5: Split Data into Training and Testing Sets ---
print("--- Step 2: Splitting data into training and test sets ---")
# stratify=y is crucial for imbalanced datasets like GTSRB
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Data shapes: X_train: {X_train.shape}, X_test: {X_test.shape}")
print("-" * 30)

# --- Block 6: Create Augmented Datasets ---
print("--- Step 3: Augmenting data with weather effects (this may take a while) ---")
# Augment the TRAINING data to teach the model
X_train_foggy = apply_effect_batch(X_train, add_fog)
X_train_rainy = apply_effect_batch(X_train, add_rain)
# Augment the TEST data to evaluate performance
X_test_foggy = apply_effect_batch(X_test, add_fog)
X_test_rainy = apply_effect_batch(X_test, add_rain)
X_test_night = apply_effect_batch(X_test, night_effect)
X_test_glare = apply_effect_batch(X_test, add_glare)
print("Data augmentation complete.")
print("-" * 30)

# --- Block 7: Combine Datasets for Robust Training ---
print("--- Step 4: Combining original and augmented training data ---")
X_train_combined = np.vstack([X_train, X_train_foggy, X_train_rainy])
y_train_combined = np.concatenate([y_train, y_train, y_train])

# Shuffle the combined dataset to ensure random order during training
shuffle_idx = np.random.permutation(len(X_train_combined))
X_train_combined = X_train_combined[shuffle_idx]
y_train_combined = y_train_combined[shuffle_idx]
print(f"Combined training data shape: {X_train_combined.shape}")
print("-" * 30)

# --- Block 8: Train the ROBUST Logistic Regression Model ---
print("--- Step 5: Training the robust model (this is the longest step) ---")
# Increased max_iter and used 'saga' solver for the large, complex dataset
lr_model = LogisticRegression(
    multi_class='multinomial', max_iter=3000, random_state=42, solver='saga', n_jobs=-1
)
lr_model.fit(X_train_combined, y_train_combined)
print("Model training complete.")
print("-" * 30)

# --- Block 9: Evaluate the New Model on All Test Conditions ---
print("--- Step 6: Evaluating the new model ---")
y_pred_normal = lr_model.predict(X_test)
y_pred_foggy = lr_model.predict(X_test_foggy)
y_pred_rainy = lr_model.predict(X_test_rainy)
y_pred_night = lr_model.predict(X_test_night)
y_pred_glare = lr_model.predict(X_test_glare)

accuracies = {
    "Normal": accuracy_score(y_test, y_pred_normal),
    "Foggy": accuracy_score(y_test, y_pred_foggy),
    "Rainy": accuracy_score(y_test, y_pred_rainy),
    "Night": accuracy_score(y_test, y_pred_night),
    "Glare": accuracy_score(y_test, y_pred_glare)
}

print("\n--- Final Robust Model Performance ---")
for condition, acc in accuracies.items():
    print(f"{condition} Accuracy: {acc:.4f}")
print("-" * 30)

# --- Block 10: Visualization ---
print("--- Step 7: Generating visualizations ---")

# Accuracy Bar Chart
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 6))
conditions = list(accuracies.keys())
scores = list(accuracies.values())
bars = ax.bar(conditions, scores, color=['#4CAF50', '#FFC107', '#2196F3', '#3F51B5', '#FF5722'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy Score', fontsize=12)
ax.set_title('Robust Logistic Regression Performance on GTSRB', fontsize=16, pad=20)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.01, f'{yval:.3f}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_normal)
plt.figure(figsize=(12, 10))
# annot=False because 43x43 is too dense for individual numbers
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=range(43), yticklabels=range(43))
plt.title('Confusion Matrix on Normal (Unaffected) Test Data', fontsize=16, pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.show()

print("\n--- Script Finished ---")